# Phase 5 — Architecture 2: LegalSeq2Seq | IA³ and LoRMA+

In [3]:
%pip -q install "tokenizers==0.21.4" "huggingface_hub==0.34.4" cloudpickle scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, accuracy_score, classification_report

BASE = Path('/kaggle/input/datasets/abdakt/a2-p3')
OUT = Path('/kaggle/working/phase5_a2_ia3_lorma')
HF_TOKEN = os.environ['HF_TOKEN']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, stride=384,
         chunks=8, chunk_batch=2, accum=8, rank=16, eval_steps=200, patience=5)
METHODS = {'ia3': dict(epochs=10, lr=5e-4), 'lorma': dict(epochs=10, lr=2e-4)}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])


## Documents and labels

In [5]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = ['Train/train.json', 'Train/dev.json', 'Test/test.json']
DOCS = {}
for name in files:
    path = hf_hub_download(REPO, 'Phase_5_LSI/'+name, repo_type='dataset',
                           revision=revision, token=HF_TOKEN)
    raw = json.loads(Path(path).read_text())
    if isinstance(raw, dict):
        raw = [raw] if 'id' in raw and 'text' in raw else [dict(v, id=k) for k,v in raw.items()]
    DOCS[Path(name).stem] = raw
labels = sorted({int(y) for split in ['train','dev'] for d in DOCS[split] for y in d['labels']})
label2id = {y:i for i,y in enumerate(labels)}
R['classes'] = len(labels)
tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
R['pad'] = tok.token_to_id('<pad>')
R['bos'] = tok.token_to_id('<bos>')

def encode_docs(docs, labelled):
    rows = []
    for doc in docs:
        ids = tok.encode(doc['text'], add_special_tokens=False).ids
        starts = list(range(0, max(1,len(ids)-R['length']+1), R['stride']))
        tail = max(0,len(ids)-R['length'])
        if starts[-1] != tail: starts.append(tail)
        if len(starts)>R['chunks']:
            starts = [starts[i] for i in np.linspace(0,len(starts)-1,R['chunks'],dtype=int)]
        chunks = [ids[s:s+R['length']] or [R['pad']] for s in starts]
        y = np.zeros(len(labels),dtype=np.float32)
        if labelled:
            for label in doc['labels']: y[label2id[int(label)]] = 1
        rows.append(dict(id=doc['id'], chunks=chunks, y=y))
    return rows

DATA = {s:encode_docs(ds,s!='test') for s,ds in DOCS.items()}
R['counts'] = np.stack([v['y'] for v in DATA['train']]).sum(0).tolist()
with (OUT/'data.pkl').open('wb') as f:
    cloudpickle.dump(dict(rows=DATA,labels=labels),f)
R['run_id'] = hashlib.sha256(json.dumps([C,R,METHODS,labels,revision,
    hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest()],sort_keys=True).encode()).hexdigest()
manifest = dict(architecture=2, config=C, settings=R, methods=METHODS, labels=labels,
                repo=REPO, revision=revision, files=files)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'tokenizer.json',OUT/'tokenizer.json')
print(' | '.join(f'{s}: {len(v):,} documents' for s,v in DATA.items())+f' | {len(labels)} statutes')


Phase_5_LSI/Train/train.json:   0%|          | 0.00/299M [00:00<?, ?B/s]

Phase_5_LSI/Train/dev.json:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Phase_5_LSI/Test/test.json:   0%|          | 0.00/98.1M [00:00<?, ?B/s]

train: 42,750 documents | dev: 10,181 documents | test: 13,019 documents | 100 statutes


## Architecture 2: LegalSeq2Seq backbone

In [6]:
# Architecture 2: LegalSeq2Seq — RoPE, RMSNorm, SwiGLU, encoder-decoder cross-attention
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-5)
        return z.to(x.dtype) * self.weight


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        self.heads = cfg['heads']
        self.head_dim = cfg['dim'] // self.heads
        self.causal = causal
        self.qkv = nn.Linear(cfg['dim'], 3 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        inv = 10000 ** (-torch.arange(0, self.head_dim, 2).float() / self.head_dim)
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos, sin = phase.cos()[None, None].to(x.dtype), phase.sin()[None, None].to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        old = 0 if past is None else past[0].size(2)
        positions = torch.arange(old, old + t, device=x.device)
        q, k = self.rope(q, positions), self.rope(k, positions)
        if past is not None:
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None if mask is None else mask[:, None, None, :]
        fast_causal = self.causal and past is None and mask is None
        if self.causal and not fast_causal:
            causal_mask = torch.arange(k.size(2), device=x.device)[None, :] <= positions[:, None]
            allowed = causal_mask[None, None] if allowed is None else allowed & causal_mask[None, None]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          is_causal=fast_causal, dropout_p=0.0)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.head_dim = cfg['heads'], cfg['dim'] // cfg['heads']
        self.q = nn.Linear(cfg['dim'], cfg['dim'], bias=False)
        self.kv = nn.Linear(cfg['dim'], 2 * cfg['dim'], bias=False)
        self.out = nn.Linear(cfg['dim'], cfg['dim'], bias=False)

    def project_memory(self, memory):
        b, s, _ = memory.shape
        k, v = self.kv(memory).view(b, s, 2, self.heads, self.head_dim).unbind(2)
        return k.transpose(1, 2), v.transpose(1, 2)

    def forward(self, x, memory, mask, cached=None):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)
        k, v = self.project_memory(memory) if cached is None else cached
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :], dropout_p=0.0)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, d))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.attn = SelfAttention(cfg, causal=False)
        self.ffn = FeedForward(cfg)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)[0]
        return x + self.ffn(self.norm2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = (RMSNorm(cfg['dim']) for _ in range(3))
        self.attn = SelfAttention(cfg, causal=True)
        self.cross = CrossAttention(cfg)
        self.ffn = FeedForward(cfg)

    def forward(self, x, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), target_mask, past, use_cache)
        x = x + a
        x = x + self.cross(self.norm2(x), memory, source_mask, cross_kv)
        return x + self.ffn(self.norm3(x)), cache


class LegalSeq2Seq(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg['encoder_layers'])])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg['decoder_layers'])])
        self.encoder_norm, self.decoder_norm = RMSNorm(cfg['dim']), RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.encoder:
            for weight in (block.attn.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(2 * len(self.encoder)))
        for block in self.decoder:
            for weight in (block.attn.out.weight, block.cross.out.weight, block.ffn.down.weight):
                nn.init.normal_(weight, std=0.02 / math.sqrt(3 * len(self.decoder)))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask):
        x = self.embed(source)
        for block in self.encoder:
            if self.training and self.cfg['gradient_checkpointing']:
                x = checkpoint(block, x, source_mask, use_reentrant=False)
            else:
                x = block(x, source_mask)
        return self.encoder_norm(x)

    def decode(self, target, memory, source_mask, target_mask=None, past=None, cross_kv=None, use_cache=False):
        x, caches = self.embed(target), []
        for i, block in enumerate(self.decoder):
            if self.training and self.cfg['gradient_checkpointing']:
                def layer(z, mem, module=block):
                    return module(z, mem, source_mask, target_mask)[0]
                x = checkpoint(layer, x, memory, use_reentrant=False)
            else:
                x, cache = block(x, memory, source_mask, target_mask,
                                 None if past is None else past[i],
                                 None if cross_kv is None else cross_kv[i], use_cache)
                caches.append(cache)
        return self.decoder_norm(x), caches


In [7]:
def features_for_chunks(lm,x,sizes,r):
    mask = torch.arange(x.size(1),device=x.device)[None]<sizes[:,None]
    h = lm.embed(x)
    for block in lm.encoder:
        h = checkpoint(block,h,mask,use_reentrant=False) if lm.training else block(h,mask)
    memory = lm.encoder_norm(h)
    query = torch.full((len(x),1),r['bos'],dtype=torch.long,device=x.device)
    h = lm.embed(query)
    for block in lm.decoder:
        h,_ = checkpoint(block,h,memory,mask,use_reentrant=False) if lm.training else block(h,memory,mask)
    decoded = lm.decoder_norm(h)[:,0]
    mean = (memory*mask[:,:,None]).sum(1)/sizes[:,None]
    return torch.cat((mean,decoded),-1)


## IA³ and LoRMA+ adapters

In [8]:
class LowRankAdapter(nn.Module):
    def __init__(self, base, rank, multiplicative=False):
        super().__init__()
        self.base = base
        self.multiplicative = multiplicative
        width = base.out_features if multiplicative else base.in_features
        self.a = nn.Parameter(torch.empty(rank,width))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.dropout = nn.Dropout(.05)
        self.scale = 2.
    def forward(self,x):
        y = self.base(x)
        h = self.dropout(x)
        if self.multiplicative:
            h = F.linear(h,self.base.weight)
        return y+self.scale*F.linear(F.linear(h,self.a),self.b)

class ActivationScale(nn.Module):
    def __init__(self,base,on_input=False):
        super().__init__()
        self.base,self.on_input = base,on_input
        self.scale = nn.Parameter(torch.ones(base.in_features if on_input else base.out_features))
    def forward(self,x):
        if self.on_input: return self.base(x*self.scale.to(x.dtype))
        y = self.base(x)
        return y*self.scale.to(y.dtype)

class SplitProjection(nn.Module):
    def __init__(self, base, sizes, names):
        super().__init__()
        self.names = names
        for name,w in zip(names,base.weight.split(sizes,0)):
            layer = nn.Linear(base.in_features,w.size(0),bias=False)
            with torch.no_grad(): layer.weight.copy_(w)
            setattr(self,name,layer)
    def forward(self,x):
        return torch.cat([getattr(self,name)(x) for name in self.names],-1)

class StatuteModel(nn.Module):
    def __init__(self,lm,r):
        super().__init__()
        self.lm, self.r = lm,r
        self.project = nn.Sequential(nn.Linear(2*lm.cfg['dim'],256),nn.Tanh(),nn.Dropout(.15))
        self.evidence = nn.Linear(256,r['classes'])
        self.attention = nn.Linear(256,r['classes'],bias=False)
        nn.init.zeros_(self.attention.weight)
    def forward(self,chunks):
        features = []
        device = next(self.parameters()).device
        for at in range(0,len(chunks),self.r['chunk_batch']):
            group = chunks[at:at+self.r['chunk_batch']]
            length = math.ceil(max(map(len,group))/8)*8
            x = torch.full((len(group),length),self.r['pad'],dtype=torch.long,device=device)
            sizes = torch.tensor([len(v) for v in group],device=device)
            for i,ids in enumerate(group): x[i,:len(ids)] = torch.tensor(ids,device=device)
            features.append(features_for_chunks(self.lm,x,sizes,self.r))
        h = self.project(torch.cat(features,0))
        weights = self.attention(h).float().softmax(0)
        return (weights*self.evidence(h).float()).sum(0)

def make_model(method,c,r):
    saved = torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
    config = saved.get('config',c).copy()
    lm = LegalSeq2Seq(config)
    lm.load_state_dict(saved['model'],strict=True)
    del saved
    d = config['dim']
    for b in list(lm.encoder)+list(lm.decoder):
        b.attn.qkv = SplitProjection(b.attn.qkv,[d,d,d],['q_proj','k_proj','v_proj'])
    for b in lm.decoder:
        b.cross.kv = SplitProjection(b.cross.kv,[d,d],['k_proj','v_proj'])
    lm.requires_grad_(False)
    for name,layer in list(lm.named_modules()):
        if not isinstance(layer,nn.Linear): continue
        key = name.split('.')[-1]
        adapted = None
        if method=='lorma' and key in ['q_proj','v_proj','out','up','down']:
            adapted = LowRankAdapter(layer,r['rank'],multiplicative=method=='lorma')
        if method=='ia3' and key in ['k_proj','v_proj','down']:
            adapted = ActivationScale(layer,on_input=key=='down')
        if adapted is not None:
            parent,key = name.rsplit('.',1)
            setattr(lm.get_submodule(parent),key,adapted)
    model = StatuteModel(lm,r)
    return model


## Prediction and calibration

In [9]:
@torch.inference_mode()
def predict(model,rows,device,r):
    model.eval()
    logits = np.empty((len(rows),r['classes']),dtype=np.float32)
    for i,row in enumerate(rows):
        with torch.autocast('cuda',dtype=torch.float16):
            logits[i] = model(row['chunks']).float().cpu().numpy()
    return logits

def probabilities(z):
    return 1/(1+np.exp(-np.clip(z,-40,40)))

def decode(p,threshold):
    pred = p>=threshold
    empty = ~pred.any(1)
    pred[np.flatnonzero(empty),p[empty].argmax(1)] = True
    return pred

def calibrate(p,gold):
    grid = np.arange(.1,.81,.05)
    scores = [f1_score(gold,decode(p,t),average='macro',zero_division=0) for t in grid]
    global_t = float(grid[int(np.argmax(scores))])
    thresholds = np.full(p.shape[1],global_t)
    for j in range(p.shape[1]):
        positives = int(gold[:,j].sum())
        if positives>=5 and positives<len(gold):
            values = [f1_score(gold[:,j],p[:,j]>=t,zero_division=0) for t in grid]
            local_t = float(grid[int(np.argmax(values))])
            weight = positives/(positives+20.)
            thresholds[j] = weight*local_t+(1-weight)*global_t
    score = f1_score(gold,decode(p,thresholds),average='macro',zero_division=0)
    if score<scores[int(np.argmax(scores))]:
        thresholds[:] = global_t
        score = max(scores)
    return thresholds,float(score)

def write_submission(rows,labels,p,threshold,path):
    chosen = decode(p,threshold)
    prediction = [dict(id=row['id'],pred=[int(labels[j]) for j in np.flatnonzero(y)])
                  for row,y in zip(rows,chosen)]
    payload = json.dumps(prediction,ensure_ascii=False,indent=2)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',payload)
    return hashlib.sha256(payload.encode()).hexdigest()


## Training

In [10]:
def run_method(gpu,method,c,r,methods):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    folder.mkdir(exist_ok=True)
    with (root/'data.pkl').open('rb') as f: data = cloudpickle.load(f)
    train,dev,test = [data['rows'][s] for s in ['train','dev','test']]
    gold = np.stack([v['y'] for v in dev])
    model = make_model(method,c,r).to(device)
    params = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
    body = [p for n,p in params if n.startswith('lm.')]
    head = [p for n,p in params if not n.startswith('lm.')]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body,lr=lr,initial_lr=lr),
                             dict(params=head,lr=3e-4,initial_lr=3e-4)],weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024)
    count = np.array(r['counts'])
    weight = np.sqrt((len(train)-count)/np.maximum(count,1)).clip(1,8)
    weight = torch.tensor(weight,dtype=torch.float32,device=device)
    nsteps = math.ceil(len(train)/r['accum'])
    total = methods[method]['epochs']*nsteps
    warmup = max(1,int(.06*total))
    best,stale,step,history = -1.,0,0,[]
    start = time.monotonic()
    total_params = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for _,p in params)
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} | {total_params:,} parameters | {trained:,} trainable',flush=True)
    logits = predict(model,dev,device,r)
    threshold,score = calibrate(probabilities(logits),gold)
    if score>best:
        best = score
        weights = {n:p.detach().cpu().float() for n,p in params}
        torch.save(dict(weights=weights,method=method,epoch=0,step=0,score=best,
                        threshold=threshold.tolist()),folder/'best.tmp')
        os.replace(folder/'best.tmp',folder/'best.pt')
        del weights
    print(f'{method} | best saved dev F1 {best:.4f}',flush=True)
    stop = False
    for epoch in range(1,methods[method]['epochs']+1):
        order = np.random.default_rng(r['seed']+epoch).permutation(len(train))
        loss_sum,seen = 0.,0
        for at in range(0,len(order),r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at+r['accum']]
            progress = max(0.,(step-warmup)/max(1,total-warmup))
            factor = min(1.,(step+1)/warmup)*(.1+.45*(1+math.cos(math.pi*progress)))
            for g in opt.param_groups: g['lr'] = g['initial_lr']*factor
            for i in group:
                y = torch.tensor(train[i]['y'],device=device)
                with torch.autocast('cuda',dtype=torch.float16):
                    z = model(train[i]['chunks'])
                    hard = F.binary_cross_entropy_with_logits(z.float(),y,pos_weight=weight)
                    loss = hard
                scaler.scale(loss/len(group)).backward()
                loss_sum += loss.detach().item()
                seen += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _,p in params],1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step%r['eval_steps']==0 or at+len(group)>=len(order):
                logits = predict(model,dev,device,r)
                threshold,score = calibrate(probabilities(logits),gold)
                improved = score>best
                stale = 0 if improved else stale+1
                if improved:
                    best = score
                    weights = {n:p.detach().cpu().float() for n,p in params}
                    torch.save(dict(weights=weights,method=method,epoch=epoch,step=step,
                                    score=best,threshold=threshold.tolist()),folder/'best.tmp')
                    os.replace(folder/'best.tmp',folder/'best.pt')
                    del weights
                    np.save(folder/'dev_logits.npy',logits)
                row = dict(epoch=epoch,step=step,loss=loss_sum/seen,macro_f1=score,stale=stale)
                history.append(row)
                (folder/'history.json').write_text(json.dumps(history,indent=2))
                print(f'{method} | epoch {epoch} step {step} | loss {row["loss"]:.4f} | dev F1 {score:.4f} | stale {stale}/5',flush=True)
                if stale>=r['patience']:
                    stop = True
                    break
        if stop: break
    print(f'{method} finished | best dev F1 {best:.4f}',flush=True)


## Train IA³ and LoRMA+ in parallel

In [11]:
def train_parallel():
    workers = []
    for gpu, method in enumerate(METHODS):
        path = OUT/f'worker_{method}.pkl'
        with path.open('wb') as f:
            cloudpickle.dump((run_method, (gpu, method, C, R, METHODS)), f)
        command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
        process = subprocess.Popen([sys.executable, '-u', '-c', command, str(path)],
                                   env=os.environ.copy(), start_new_session=True)
        workers.append(process)
        print(f'{method} starting on GPU {gpu}', flush=True)
    for process in workers:
        process.wait()

train_parallel()

ia3 starting on GPU 0
lorma starting on GPU 1
ia3 | GPU 0 | 176,132,452 parameters | 524,644 trainable
lorma | GPU 1 | 179,493,220 parameters | 3,885,412 trainable
ia3 | best saved dev F1 0.0749
lorma | best saved dev F1 0.0729
ia3 | epoch 1 step 200 | loss 0.7168 | dev F1 0.0752 | stale 0/5
lorma | epoch 1 step 200 | loss 0.6652 | dev F1 0.0731 | stale 0/5
ia3 | epoch 1 step 400 | loss 0.6068 | dev F1 0.0756 | stale 0/5
ia3 | epoch 1 step 600 | loss 0.5512 | dev F1 0.0775 | stale 0/5
lorma | epoch 1 step 400 | loss 0.5724 | dev F1 0.0719 | stale 1/5
ia3 | epoch 1 step 800 | loss 0.5204 | dev F1 0.0824 | stale 0/5
lorma | epoch 1 step 600 | loss 0.5279 | dev F1 0.0704 | stale 2/5
ia3 | epoch 1 step 1000 | loss 0.5013 | dev F1 0.0990 | stale 0/5
ia3 | epoch 1 step 1200 | loss 0.4883 | dev F1 0.1171 | stale 0/5
lorma | epoch 1 step 800 | loss 0.5028 | dev F1 0.1076 | stale 0/5
ia3 | epoch 1 step 1400 | loss 0.4768 | dev F1 0.1372 | stale 0/5
ia3 | epoch 1 step 1600 | loss 0.4666 | dev F1

## Export submissions

In [ ]:
from IPython.display import FileLink, display
with (OUT/'data.pkl').open('rb') as f:
    data = cloudpickle.load(f)
sub = OUT/'submissions'
sub.mkdir(exist_ok=True)
scores = {}
for method in METHODS:
    gc.collect()
    torch.cuda.empty_cache()
    saved = torch.load(OUT/method/'best.pt', map_location='cpu', weights_only=False)
    model = make_model(method, C, R)
    model.load_state_dict(saved['weights'], strict=False)
    model = model.to('cuda:0')
    logits = predict(model, data['rows']['test'], torch.device('cuda:0'), R)
    path = sub/f'submission_A2_{method}.zip'
    write_submission(data['rows']['test'], data['labels'], probabilities(logits),
                     np.array(saved['threshold']), path)
    scores[method] = saved['score']
    print(f"{method} | saved epoch {saved['epoch']} step {saved['step']} | dev F1 {saved['score']:.4f}")
    display(FileLink(str(path)))
    del model, saved
best = max(scores, key=scores.get)
path = sub/'submission_A2_best.zip'
shutil.copyfile(sub/f'submission_A2_{best}.zip', path)
print(f'Best method: {best}')
display(FileLink(str(path)))
gc.collect()
torch.cuda.empty_cache()

ia3 | saved epoch 1 step 3400 | dev F1 0.2362


/kaggle/working/phase5_a2_ia3_lorma/submissions/submission_A2_ia3.zip

## References